In [15]:

import pandas as pd
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")


import sys
sys.path.append("../")

from src.data.clean_data import (
    run_pre_cleaning_checks, select_modeling_columns, inspect_total_charges_outliers
)
from src.pipeline.preprocessing_pipeline import build_preprocessing_pipeline
from sklearn.model_selection import train_test_split
import joblib

In [16]:
# 1. Quality checks
print(run_pre_cleaning_checks(train, "train"))
print(run_pre_cleaning_checks(test, "test"))


{'name': 'train', 'avg_missing_pct': np.float64(0.0), 'n_duplicate_rows': 0, 'n_duplicated_ids': 0}
{'name': 'test', 'avg_missing_pct': np.float64(0.0), 'n_duplicate_rows': 0, 'n_duplicated_ids': 0}


In [17]:
# 2. Column selection
train_selected = select_modeling_columns(train, include_target=True)
test_selected = select_modeling_columns(test, include_target=False)


In [18]:
# 3. Outlier inspection (interpretation stays here, in the notebook)
outliers = inspect_total_charges_outliers(train_selected)
print(f"Outliers: {len(outliers)} | Avg AccountAge: {outliers['AccountAge'].mean():.1f}")

Outliers: 741 | Avg AccountAge: 116.9


In [19]:
# 4. Split
X = train_selected.drop(columns=["Churn"])
y = train_selected["Churn"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [20]:
# 5. Build + fit pipeline
preprocessor = build_preprocessing_pipeline()
X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(test_selected)

In [21]:
# 6. Save
joblib.dump(preprocessor, "../models/preprocessing_pipeline.pkl")
train_selected.to_csv("../data/processed/train_clean.csv", index=False)
test_selected.to_csv("../data/processed/test_clean.csv", index=False)